In [4]:
!pip install streamlit langchain-groq
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [5]:
%%writefile app.py
import streamlit as st


st.title('🧠The Memory Test')


col1, col2 = st.columns(2)


with col1:
    st.subheader('Normal Variable')


    normal_counter = 0


    if st.button("Click Me (Normal)"):
        normal_counter = normal_counter + 1


    st.write(f'Normal Counter: {normal_counter}')


with col2:
    st.subheader('Session State')


    if 'counter' not in st.session_state: # Will only work when we first time open the website
        st.session_state.counter = 0


    if st.button('Click Me(Remember)'):
        st.session_state.counter = st.session_state.counter + 1


    st.write(f'Memory Counter: {st.session_state.counter}')


Overwriting app.py


In [6]:
!pip install pyngrok -q

In [7]:
%%writefile app.py
import streamlit as st


st.title('💬 The Echo Bot (Memory Test)')


# 1. Initialise a chat history
# We will store a list of dictionaries, just like we do in langchain
if 'messages' not in st.session_state:
    st.session_state.messages = [] # Only run for first time, later it will store


# 2. Display the chat history
# Every time somebody clicks a button or something, we still have all the old messages
for msg in st.session_state.messages:
    with st.chat_messages(msg['role']):
        st.markdown(msg['content'])


# 3. The Input Box (Pinned to the bottom of the screen)
# The 'walrus operator' (:=) assigns the input to 'user_query' AND checks if it's not empty simultaneously.
if user_query := st.chat_input('Say something to the Echo bot.....'):


    with st.chat_messages('user'):
        st.markdown(user_query)


    st.session_state.messages.append({'role':'user', 'content':user_query})
    bot_response = f"Echo: You just said '{user_query}'"


    with st.chat_messages('assistant'):
        st.markdown(bot_response)



    st.session_state.messages.append({'role':'assistant', 'content':bot_response})

Overwriting app.py


In [8]:
%%writefile app.py
import streamlit as st


st.title('💬 The Echo Bot (Memory Test)')


# 1. Initialise a chat history
# We will store a list of dictionaries, just like we do in langchain
if 'messages' not in st.session_state:
    st.session_state.messages = [] # Only run for first time, later it will store


# 2. Display the chat history
# Every time somebody clicks a button or something, we still have all the old messages
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])


# 3. The Input Box (Pinned to the bottom of the screen)
# The 'walrus operator' (:=) assigns the input to 'user_query' AND checks if it's not empty simultaneously.
if user_query := st.chat_input('Say something to the Echo bot.....'):


    with st.chat_message('user'):
        st.markdown(user_query)


    st.session_state.messages.append({'role':'user', 'content':user_query})


    bot_response = f"Echo: You just said '{user_query}'"


    with st.chat_message('assistant'):
      st.markdown(bot_response)



    st.session_state.messages.append({'role':'assistant', 'content':bot_response})



Overwriting app.py


In [9]:
from pyngrok import ngrok


ngrok.kill()

In [10]:
%%writefile app.py
import streamlit as st
import json
import os
import uuid

st.title("👋 The Name Taker")

# File to store names (persists in Colab)
NAME_FILE = "user_names.json"

# Load saved names
if os.path.exists(NAME_FILE):
    with open(NAME_FILE, 'r') as f:
        saved_names = json.load(f)
else:
    saved_names = {}

# Use query params to persist across browser refreshes
query_params = st.query_params

if "user_id" not in query_params:
    # New browser session - generate ID and add to URL
    user_id = str(uuid.uuid4())[:8]
    st.query_params["user_id"] = user_id
else:
    user_id = query_params["user_id"][0] if isinstance(query_params["user_id"], list) else query_params["user_id"]

st.sidebar.write(f"User ID: {user_id}")

# Initialize session state
if "user_name" not in st.session_state:
    if user_id in saved_names:
        # Returning user - load from database
        st.session_state.user_name = saved_names[user_id]
        st.session_state.is_returning = True
    else:
        st.session_state.is_returning = False

# Main logic
if "user_name" not in st.session_state:
    # New user - ask for name
    st.subheader("Before we start chatting, what's your name?")
    name = st.text_input("Enter your name:")

    if st.button("Submit"):
        if name:
            st.session_state.user_name = name
            saved_names[user_id] = name
            with open(NAME_FILE, 'w') as f:
                json.dump(saved_names, f)
            st.rerun()
        else:
            st.warning("Please enter a name!")
else:
    # User exists
    if st.session_state.get("is_returning", False):
        # This is a returning user (found in database)
        st.title(f"✨ Welcome back, {st.session_state.user_name}! ✨")
    else:
        # First time user
        st.title(f"✨ Hello {st.session_state.user_name}, what can I do for you today? ✨")
        # Mark as not returning for next time
        st.session_state.is_returning = True

Overwriting app.py


In [11]:
from pyngrok import ngrok
import subprocess
import time


ngrok.set_auth_token("3Al7vXYOerrZOoGDXDZAZGQ1Ge7_rBMhCMmz9f6Pg7BBSCot")


# Start streamlit as a background process (doesn't block)
subprocess.Popen(["streamlit", "run", "app.py"])


# Wait for Streamlit to fully start
time.sleep(5)


# Connect ngrok
public_url = ngrok.connect(8501)
print("✅ Your app is live at:", public_url)

✅ Your app is live at: NgrokTunnel: "https://premodern-isleless-refugia.ngrok-free.dev" -> "http://localhost:8501"


In [12]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

In [13]:
llm = ChatGroq (
    temperature = 0.7,
    model = 'llama-3.1-8b-instant',
    api_key = userdata.get('groq_api')
)

In [14]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq

st.set_page_config(page_title='groq_api', layout='centered')
st.title('🤖 The Groq Chatbot')
st.write('A fully integrated, memory enabled AI Assistant')

# The Sidebar
with st.sidebar:
    st.header('⚙️Configuration')
    user_api_key = st.text_input('Enter Groq API Key:', type='password')
    st.info('Your key is required to wake the AI Brain')

# The Memory
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display the History
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):  # Fixed: chat_message (singular)
        st.markdown(msg['content'])

# Fix: Added colon and fixed indentation
if user_query := st.chat_input('Message the AI....'):
    if not user_api_key:
        st.error("Please enter your API Key in the sidebar first!")
    else:
        st.session_state.messages.append({'role':'user', 'content':user_query})
        with st.chat_message('user'):  # Fixed: chat_message (singular)
            st.markdown(user_query)

        llm = ChatGroq(
            temperature=0.7,
            model='llama-3.3-70b-versatile',
            api_key=user_api_key
        )

        with st.spinner('AI is thinking...'):  # Fixed: added colon
            response = llm.invoke(st.session_state.messages)
            bot_answer = response.content

        # Fixed: session_state (not session_stat) and messages.append
        st.session_state.messages.append({'role':'assistant', 'content':bot_answer})
        with st.chat_message('assistant'):  # Fixed: chat_message (singular)
            st.markdown(bot_answer)

Overwriting app.py


In [15]:
# Practice Challenge: "The System Prompt Injector"
# Scenario: You have a working chatbot, but it is generic. You want to give the user the ability to define the AI's "Persona" (e.g., "You are a pirate" or "You are a strict math teacher") using a sidebar text area.


# The Logic Puzzle: LangChain expects the System Prompt to be the very first message in the history. We need to insert a {"role": "system", "content": persona} into the vault, but we only want to do it once!


# Your Task:


# In the sidebar, add persona = st.text_area("System Prompt:", value="You are a helpful assistant.").


# Add a button in the sidebar called "Reset Chat & Apply Persona".


# If that button is clicked, clear the vault (st.session_state.messages = []) so the chat restarts with the new persona.


# Before llm.invoke() happens, ensure the first message sent to the LLM includes the persona.

%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq

st.set_page_config(page_title='groq_api', layout='centered')
st.title('🤖 The Groq Chatbot')
st.write('A fully integrated, memory enabled AI Assistant')

# The Sidebar
with st.sidebar:
    st.header('⚙️Configuration')
    user_api_key = st.text_input('Enter Groq API Key:', type='password')
    st.info('Your key is required to wake the AI Brain')

    st.divider()  # Add a visual separator

    # NEW: System Prompt Injector
    st.header('🎭 AI Persona')
    persona = st.text_area(
        "System Prompt:",
        value="You are a helpful assistant.",
        help="Define the AI's personality and behavior"
    )

    # NEW: Reset button
    if st.button("🔄 Reset Chat & Apply Persona", use_container_width=True):
        st.session_state.messages = []  # Clear the vault
        st.session_state.persona_applied = False  # Reset the flag
        st.rerun()

# The Memory
if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.persona_applied = False

# Display the History
for msg in st.session_state.messages:
    # Skip displaying system messages in chat UI
    if msg['role'] != 'system':
        with st.chat_message(msg['role']):
            st.markdown(msg['content'])

# Handle user input
if user_query := st.chat_input('Message the AI....'):
    if not user_api_key:
        st.error("Please enter your API Key in the sidebar first!")
    else:
        # Add user message to session state
        st.session_state.messages.append({'role': 'user', 'content': user_query})
        with st.chat_message('user'):
            st.markdown(user_query)

        # Initialize LLM
        llm = ChatGroq(
            temperature=0.7,
            model='llama-3.3-70b-versatile',
            api_key=user_api_key
        )

        # Prepare messages for LLM (including system prompt if needed)
        messages_for_llm = []

        # Add system prompt if not already applied
        if not st.session_state.persona_applied:
            messages_for_llm.append({'role': 'system', 'content': persona})
            st.session_state.persona_applied = True

        # Add all conversation history (excluding any old system messages)
        for msg in st.session_state.messages:
            if msg['role'] != 'system':  # Skip any old system messages
                messages_for_llm.append(msg)

        # Get AI response
        with st.spinner('AI is thinking...'):
            response = llm.invoke(messages_for_llm)
            bot_answer = response.content

        # Add assistant response to session state
        st.session_state.messages.append({'role': 'assistant', 'content': bot_answer})
        with st.chat_message('assistant'):
            st.markdown(bot_answer)

Overwriting app.py


In [16]:
# Install required packages
!pip install langchain langchain-community langchain-groq langgraph duckduckgo-search

In [17]:
# Install all required packages including ddgs
!pip install langchain langchain-community langchain-groq langgraph duckduckgo-search ddgs

In [18]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage


st.set_page_config(page_title="Live Web Agent", layout="centered")


st.title("🌍 The Live Internet Agent")
st.write("Ask me anything about current events. I will browse the web to find the answer.")


# --- 1. SIDEBAR CONFIGURATION ---
with st.sidebar:
    st.header("⚙️ System Config")
    user_api_key = st.text_input("Groq API Key:", type="password")
    st.info("Equipped with: DuckDuckGo Web Search Tool")


# --- 2. THE MEMORY VAULT ---
if "messages" not in st.session_state:
    st.session_state.messages = []


# --- 3. DRAW THE CHAT HISTORY ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])


# --- 4. THE CORE AGENTIC LOOP ---
if user_query := st.chat_input("Ask about today's news..."):

    if not user_api_key:
        st.error("Please enter your API Key in the sidebar.")
    else:
        # A. Display User Message
        st.session_state.messages.append({"role": "user", "content": user_query})
        with st.chat_message("user"):
            st.markdown(user_query)


        # B. Initialize the LangGraph Agent Engine
        llm = ChatGroq(temperature=0, model_name="llama3-8b-8192", api_key=user_api_key)
        web_tool = DuckDuckGoSearchRun()

        # We define a strict system prompt to ensure it actually uses the tool!
        sys_prompt = "You are a live research assistant. You MUST use the web search tool to find current information before answering. Synthesize the results cleanly."

        agent = create_react_agent(llm, tools=[web_tool], state_modifier=sys_prompt)


        # C. THE BRIDGE: Translate Streamlit Memory -> LangGraph Memory
        langgraph_history = []
        for m in st.session_state.messages:
            if m["role"] == "user":
                langgraph_history.append(HumanMessage(content=m["content"]))
            else:
                langgraph_history.append(AIMessage(content=m["content"]))


        # D. Execute the Agent (With a visual loading spinner)
        with st.chat_message("assistant"):
            with st.spinner("🤖 Browsing the web and analyzing results..."):

                # We feed the translated history into the graph
                result_state = agent.invoke({"messages": langgraph_history})

                # The final answer is the last message in the state
                bot_answer = result_state["messages"][-1].content

            # Display the final answer on the UI
            st.markdown(bot_answer)

        # E. Save the final answer back to Streamlit's Vault
        st.session_state.messages.append({"role": "assistant", "content": bot_answer})


Overwriting app.py


In [19]:
# Nuclear option - resets everything
!pkill -f streamlit
!pkill -f ngrok

from pyngrok import ngrok
import subprocess
import time
from google.colab import userdata

ngrok.kill()
ngrok.set_auth_token(userdata.get('ngrok_auth_token'))

subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(8)

public_url = ngrok.connect(8501)
print("✅ App is live at:", public_url)

✅ App is live at: NgrokTunnel: "https://premodern-isleless-refugia.ngrok-free.dev" -> "http://localhost:8501"


In [20]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver

st.set_page_config(page_title="Live Web Agent", layout="centered")

st.title("🌍 The Live Internet Agent")
st.write("Ask me anything about current events. I will browse the web to find the answer.")

# --- 1. SIDEBAR CONFIGURATION ---
with st.sidebar:
    st.header("⚙️ System Config")
    user_api_key = st.text_input("Groq API Key:", type="password")
    st.info("Equipped with: DuckDuckGo Web Search Tool")

    st.divider()
    st.header("🤖 Model Selection")
    model_option = st.selectbox(
        "Choose a model:",
        [
            "llama-3.3-70b-versatile",  # Latest Llama 3.3
            "llama-3.1-8b-instant",     # Fast Llama 3.1 8B
            "mixtral-8x7b-32768",       # Mixtral
            "gemma2-9b-it"              # Google's Gemma 2
        ],
        index=0
    )

# --- 2. THE MEMORY VAULT ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# --- 3. DRAW THE CHAT HISTORY ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- 4. THE CORE AGENTIC LOOP ---
if user_query := st.chat_input("Ask about today's news..."):

    if not user_api_key:
        st.error("Please enter your API Key in the sidebar.")
    else:
        # A. Display User Message
        st.session_state.messages.append({"role": "user", "content": user_query})
        with st.chat_message("user"):
            st.markdown(user_query)

        # B. Initialize the LangGraph Agent Engine
        llm = ChatGroq(
            temperature=0,
            model_name=model_option,
            api_key=user_api_key
        )

        # Create the search tool
        web_tool = DuckDuckGoSearchRun()

        # Define a strict system prompt
        sys_prompt = """You are a live research assistant.
        You have access to a web search tool.
        You MUST use the web search tool to find current information before answering.

        When you receive a question:
        1. First, use the web search tool to find relevant information
        2. Then, synthesize the results into a clear, accurate answer
        3. Always cite your sources when possible

        Remember: Always search first, then answer based on the search results."""

        # Create the agent with proper configuration
        agent = create_react_agent(
            llm,
            tools=[web_tool],
            prompt=sys_prompt,
            checkpointer=MemorySaver()  # Add memory saver
        )

        # C. Translate Streamlit Memory -> LangGraph Memory with proper format
        langgraph_messages = []

        # Add system message first
        langgraph_messages.append(SystemMessage(content=sys_prompt))

        # Add conversation history
        for m in st.session_state.messages:
            if m["role"] == "user":
                langgraph_messages.append(HumanMessage(content=m["content"]))
            else:
                langgraph_messages.append(AIMessage(content=m["content"]))

        # D. Execute the Agent
        with st.chat_message("assistant"):
            with st.spinner(f"🤖 Searching the web for answers..."):
                try:
                    # Feed the messages into the graph
                    result_state = agent.invoke({
                        "messages": langgraph_messages
                    }, config={"configurable": {"thread_id": "1"}})  # Add thread_id for memory

                    # Extract the final answer
                    final_messages = result_state["messages"]
                    bot_answer = final_messages[-1].content if final_messages else "No answer found."

                except Exception as e:
                    # Fallback to simple search if agent fails
                    try:
                        search_result = web_tool.run(user_query)
                        bot_answer = f"**Search Results:**\n\n{search_result}"
                    except:
                        bot_answer = f"I encountered an error: {str(e)}"

            st.markdown(bot_answer)

        # E. Save the final answer
        st.session_state.messages.append({"role": "assistant", "content": bot_answer})

Overwriting app.py


In [21]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver

st.set_page_config(page_title="Live Web Agent", layout="centered")

st.title("🌍 The Live Internet Agent")
st.write("Ask me anything about current events. I will browse the web to find the answer.")

# --- 1. SIDEBAR CONFIGURATION ---
with st.sidebar:
    st.header("⚙️ System Config")
    user_api_key = st.text_input("Groq API Key:", type="password")
    st.info("Equipped with: DuckDuckGo Web Search Tool")

    st.divider()
    st.header("🤖 Model Selection")
    model_option = st.selectbox(
        "Choose a model:",
        [
            "llama-3.3-70b-versatile",  # Latest Llama 3.3
            "llama-3.1-8b-instant",     # Fast Llama 3.1 8B
            "mixtral-8x7b-32768",       # Mixtral
            "gemma2-9b-it"              # Google's Gemma 2
        ],
        index=0
    )

    st.divider()
    st.header("🛠️ Tool Control")
    # NEW: Toggle switch for web search
    use_search = st.toggle("Enable Web Search", value=True)
    if use_search:
        st.success("🌐 Web Search is **ON** - I'll search the internet for answers")
    else:
        st.info("💬 Web Search is **OFF** - I'll answer from my knowledge only")

# --- 2. THE MEMORY VAULT ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# --- 3. DRAW THE CHAT HISTORY ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- 4. THE CORE AGENTIC LOOP ---
if user_query := st.chat_input("Ask about today's news..."):

    if not user_api_key:
        st.error("Please enter your API Key in the sidebar.")
    else:
        # A. Display User Message
        st.session_state.messages.append({"role": "user", "content": user_query})
        with st.chat_message("user"):
            st.markdown(user_query)

        # B. Initialize the LangGraph Agent Engine
        llm = ChatGroq(
            temperature=0,
            model_name=model_option,
            api_key=user_api_key
        )

        # Create the search tool
        web_tool = DuckDuckGoSearchRun()

        # NEW: Tool toggle logic - set active tools based on toggle
        if use_search:
            active_tools = [web_tool]
            tool_status = "with web search enabled"
        else:
            active_tools = []  # Empty list = no tools, just chat
            tool_status = "with web search disabled (knowledge only)"

        # Define system prompt - modified based on tool availability
        if use_search:
            sys_prompt = """You are a live research assistant.
            You have access to a web search tool.
            You MUST use the web search tool to find current information before answering.

            When you receive a question:
            1. First, use the web search tool to find relevant information
            2. Then, synthesize the results into a clear, accurate answer
            3. Always cite your sources when possible

            Remember: Always search first, then answer based on the search results."""
        else:
            sys_prompt = """You are a helpful AI assistant.
            You do not have access to web search in this mode.
            Answer questions based on your training knowledge only.
            If you don't know something, simply say so."""

        # Create the agent with active_tools (which may be empty)
        agent = create_react_agent(
            llm,
            tools=active_tools,  # This can be [] if search is disabled
            prompt=sys_prompt,
            checkpointer=MemorySaver()
        )

        # C. Translate Streamlit Memory -> LangGraph Memory
        langgraph_messages = []
        langgraph_messages.append(SystemMessage(content=sys_prompt))

        for m in st.session_state.messages:
            if m["role"] == "user":
                langgraph_messages.append(HumanMessage(content=m["content"]))
            else:
                langgraph_messages.append(AIMessage(content=m["content"]))

        # D. Execute the Agent
        with st.chat_message("assistant"):
            with st.spinner(f"🤖 Thinking {tool_status}..."):
                try:
                    result_state = agent.invoke({
                        "messages": langgraph_messages
                    }, config={"configurable": {"thread_id": "1"}})

                    final_messages = result_state["messages"]
                    bot_answer = final_messages[-1].content if final_messages else "No answer found."

                except Exception as e:
                    # Fallback to simple search ONLY if search is enabled
                    if use_search:
                        try:
                            search_result = web_tool.run(user_query)
                            bot_answer = f"**Search Results:**\n\n{search_result}"
                        except:
                            bot_answer = f"I encountered an error: {str(e)}"
                    else:
                        bot_answer = f"I encountered an error: {str(e)}"

            st.markdown(bot_answer)

        # E. Save the final answer
        st.session_state.messages.append({"role": "assistant", "content": bot_answer})

Overwriting app.py


In [22]:
from google.colab import userdata
ngrok_auth_token = userdata.get('ngrok_auth_token')

from pyngrok import ngrok
import subprocess
import time


ngrok.set_auth_token(ngrok_auth_token)


# Kill any existing tunnels
ngrok.kill()


# Start streamlit as a background process
subprocess.Popen(["streamlit", "run", "app.py"])


# Wait for Streamlit to fully start
time.sleep(5)


# Connect ngrok with a fresh random URL (no saved domain)
public_url = ngrok.connect(8501, bind_tls=True)
print("✅ Your app is live at:", public_url)


✅ Your app is live at: NgrokTunnel: "https://premodern-isleless-refugia.ngrok-free.dev" -> "http://localhost:8501"
